### Models for Milestone 3

In [41]:
#Librariesfrom sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, recall_score, roc_auc_score
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
import pandas as pd
import numpy as np

In [37]:
def categorize_conflict(x, q1, q3):
    if x == 0:
        return 0
    elif x <= q1:
        return 1
    elif x <= q3:
        return 2
    else:
        return 3

In [38]:
parquet_path = '../data/output/grid_conflict_climate_2019_23.parquet'

#Drop nas #
df = pd.read_parquet(parquet_path)
df = df.dropna()
new_df = df[df["conflict_count"] > 0]
q1, q2, q3 = new_df['conflict_count'].quantile(0.25), new_df['conflict_count'].quantile(0.50), new_df['conflict_count'].quantile(0.75)

df['target'] = df['conflict_count'].apply(lambda x: categorize_conflict(x, q1, q3))
features = df.drop(['GEOID', 'conflict_count', 'target'], axis=1)
features = pd.get_dummies(features, columns=['year'], prefix='year')
X = features
y = df['target']

In [43]:

# Preprocess data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train with gradient descent
# 'saga' is a variant of stochastic gradient descent

param_grid = {
    'estimator__C': np.logspace(-3, 2, 6),           # C: [0.001, 0.01, 0.1, 1, 10, 100]
    'estimator__l1_ratio': np.linspace(0, 1, 5)      # l1_ratio: [0, 0.25, 0.5, 0.75, 1]
}

base_model = LogisticRegression(
    solver='saga',
    penalty='elasticnet',
    max_iter=1000,
    random_state=42
)

ovr_model = OneVsRestClassifier(base_model)

grid = GridSearchCV(
    ovr_model,
    param_grid,
    cv=5,
    scoring='recall_weighted',  # or another multilabel-compatible metric
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

# Results
print("Best C:", grid.best_params_['estimator__C'])
print("Best l1_ratio:", grid.best_params_['estimator__l1_ratio'])
print("Best score:", grid.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best C: 0.1
Best l1_ratio: 0.75
Best score: 0.9232691102328546


In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(len(X_test))

6803


### Concerned about class imbalance

In [46]:

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Base model
base_model = LogisticRegression(
    solver='saga',
    penalty='elasticnet',
    max_iter=1000,
    random_state=42
)

# Grid of hyperparameters
param_grid = {
    'C': np.logspace(-3, 1, 5),            # [0.001, 0.01, 0.1, 1, 10]
    'l1_ratio': np.linspace(0, 1, 5)       # [0.0, 0.25, 0.5, 0.75, 1.0]
}

# Grid search with recall_weighted
grid = GridSearchCV(
    base_model,
    param_grid,
    scoring='recall_weighted',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit grid search
grid.fit(X_train_smote, y_train_smote)

# Evaluate
best_model = grid.best_estimator_
y_pred_smote = best_model.predict(X_test_scaled)

print("\n--- Tuned SMOTE + ElasticNet Model ---")
print(f"Best C: {grid.best_params_['C']}")
print(f"Best l1_ratio: {grid.best_params_['l1_ratio']}")
print(f"Best weighted recall (CV): {grid.best_score_:.4f}")

labels = best_model.classes_

# Confusion matrix with labels
conf_matrix = confusion_matrix(y_test, y_pred_smote)
conf_df = pd.DataFrame(conf_matrix, index=labels, columns=labels)

print(f"\nTest Accuracy: {best_model.score(X_test_scaled, y_test):.4f}")
print(f"Test Recall (weighted): {recall_score(y_test, y_pred_smote, average='weighted'):.4f}")
print("\nConfusion Matrix:")
print(conf_df)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))


Fitting 5 folds for each of 25 candidates, totalling 125 fits


/Users/agustin.ep/Library/CloudStorage/OneDrive-TheUniversityofChicago/01 UChicago/04 Spring 2025/02 ML/Cyborg Paul/project-aeyzaguirre-phernandezpedraz-afcamachob/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/agustin.ep/Library/CloudStorage/OneDrive-TheUniversityofChicago/01 UChicago/04 Spring 2025/02 ML/Cyborg Paul/project-aeyzaguirre-phernandezpedraz-afcamachob/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/agustin.ep/Library/CloudStorage/OneDrive-TheUniversityofChicago/01 UChicago/04 Spring 2025/02 ML/Cyborg Paul/project-aeyzaguirre-phernandezpedraz-afcamachob/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn


--- Tuned SMOTE + ElasticNet Model ---
Best C: 10.0
Best l1_ratio: 0.75
Best weighted recall (CV): 0.5215

Test Accuracy: 0.8064
Test Recall (weighted): 0.8064

Confusion Matrix:
      0    1    2    3
0  5302  567  253  129
1    81   78   39   18
2    33   70   48   52
3    15   25   35   58

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.85      0.91      6251
           1       0.11      0.36      0.16       216
           2       0.13      0.24      0.17       203
           3       0.23      0.44      0.30       133

    accuracy                           0.81      6803
   macro avg       0.36      0.47      0.38      6803
weighted avg       0.91      0.81      0.85      6803

